# MOLTIE – Jupyter Inference Cell (y_spec + Needle Runner)

## Purpose

This notebook cell executes the `y_spec.py` schema inference module **and** a closed-set needle classifier over each row of a witness statement CSV file. It is designed for **interactive debugging, enrichment, and calibration**, not bulk production execution.

It enables row-level inspection of semantic tagging (needles), structured Y inference, JSON validity, and stability before transitioning to a production batch workflow.

---

## Input

* **CSV file**:
  `/home/hello/Projects/Statements/input/Leonardo_WS.csv`

* **Column used**:
  `text_verbatim`

Each row from `text_verbatim` is treated as `ws_text` and:

1. Passed via `stdin` to `y_spec.py`
2. Independently evaluated by the needle classifier (LLM closed-set tagging)

---

## Outputs

Two files are written to:

`/home/hello/Projects/Statements/output`

1. **Leonardo_WS_enhanced.csv**

   * Contains **all original CSV columns**
   * Adds:

     * `needle_selected_raw`
     * `has__/conf__/quote__` columns per tag
     * `y_ok`, `y_rc`
     * `ws_len`
     * `X1` (1-based row id)
     * `doc`

2. **Y_inferred.json**

   * Aggregates structured `y_spec` output per processed row
   * Preserves full Y JSON for auditability

---

## Execution Flow

### 1. Configuration

The cell defines:

* Model name (`mistral-small3.2:latest`)
* Path to `y_spec.py`
* Debug mode toggle
* Flexible row selector (`DEBUG_SLICE`)
* Needle tag taxonomy (closed set)

Debug selector supports:

* `"3"` → process exactly the 3rd row (1-based)
* `":5"` → first 5 rows
* `"2:5"` → Python slice semantics

This allows precise surgical debugging.

---

### 2. CSV Loading

* Reads the CSV using pandas
* Validates `text_verbatim` exists
* Converts nulls to empty strings
* Determines which rows to process (full run or debug slice)

Original CSV structure is preserved.

---

### 3. Row Iteration (with tqdm)

For each selected row:

* Assigns a deterministic 1-based identifier (`X1`)
* Strips whitespace
* Skips empty rows
* Prints trace:

```
X1=<row_number> doc=Leonardo_WS.csv
```

This guarantees reproducibility and traceability.

---

### 4. Dual Processing Per Row

Each `ws_text` flows through two independent pipes:

#### A) Needle Classifier (Semantic Tagging)

* Closed-set LLM classification
* Evidence-quoted
* Negation-aware
* JSON-validated
* Returns:

  * Selected tags
  * Confidence scores
  * Evidence quotes

Expanded into structured columns (`has__/conf__/quote__`).

#### B) Y-Spec Structured Inference

```
python y_spec.py --model mistral-small3.2:latest
```

* Input via `stdin`
* Captures:

  * `stdout`
  * `stderr`
  * `returncode`
* Attempts immediate JSON parsing
* Aggregates valid results into `Y_inferred.json`

Needle tagging and Y inference remain structurally independent.

---

### 5. Enrichment Merge

The enrichment fields are merged back into the **full original DataFrame**, ensuring:

* No original data is lost
* Only processed rows receive populated enrichment fields (in debug mode)
* Non-processed rows remain intact

---

## Why This Design Is Intentional

This notebook version prioritises:

* Structural independence between retrieval (needles) and reasoning (Y)
* Full preservation of source data
* Evidence-anchored tagging
* Deterministic debug slicing
* Immediate JSON validation
* Transparent failure surfaces

It does **not** parallelise.

It does **not** bias X/Y extraction toward needle concepts.

It maintains clean architectural separation between semantic tagging and structured inference.

---

## When to Transition to Production Script

Move to a full `.py` batch runner once:

* Needle classification is stable
* Y JSON schema is consistent
* Debug slicing no longer required
* Error patterns are understood

Production version should then:

* Support parallel execution
* Implement retry logic
* Log failures explicitly
* Optionally write per-row Y JSON files

---

## Role in the MOLTIE Architecture

This cell functions as a **calibration and enrichment harness**.

It sits between:

* Raw witness statement substrate
* Independent semantic tagging layer
* Structured Y inference layer

It ensures that:

* Retrieval signals (needles)
* Structural reasoning signals (Y)

are derived independently from the same text.

---

## Summary

This Jupyter cell provides a controlled inference environment that:

* Reads witness statement rows
* Applies independent needle classification
* Executes `y_spec.py`
* Validates structured JSON output
* Merges enrichment into the original dataset
* Writes consolidated outputs

It is modular, auditable, deterministic under debug, and structurally clean.


In [5]:
import json
import subprocess
import time
import re
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

# =========================================================
# CONFIG
# =========================================================
CSV_PATH = Path("/home/hello/Projects/Statements/input/Leonardo_WS.csv")
TEXT_COL = "text_verbatim"

OUT_DIR = Path("/home/hello/Projects/Statements/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ENHANCED_CSV = OUT_DIR / "Leonardo_WS_enhanced.csv"
OUT_Y_JSON = OUT_DIR / "Y_inferred.json"

Y_SPEC_PY = Path("/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py")
MODEL = "mistral-small3.2:latest"
OLLAMA_CMD = ["ollama", "run", MODEL]

DEBUG = False
DEBUG_SLICE = "3"

# Timeouts (seconds) — tune as you like
NEEDLE_TIMEOUT = 180
Y_SPEC_TIMEOUT = 180

# Needle tags (closed set)
# NOTE: keep these as the canonical needle vocabulary.
ALLOWED_TAGS = [
    "upheld",
    "verbal_warning",
    "no_contemporaneous_evidence",
    "predetermination",
    "appeal_scope_limitation",
    "none",
]
TAG_DEFS = {
    "upheld": "appeal upheld / allowed / succeeds",
    "verbal_warning": "verbal warning mentioned as a disciplinary step or fact",
    "no_contemporaneous_evidence": "absence of notes/records or finding of no contemporaneous evidence",
    "predetermination": "decision appeared predetermined / outcome fixed / mind closed",
    "appeal_scope_limitation": "point not in grounds / outside scope / refused/declined to consider / limited to grounds",
    "none": "none of the above apply",
}

# =========================================================
# HELPERS
# =========================================================
def _extract_json_object(raw: str) -> str:
    raw = (raw or "").strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object detected in output.")
    return raw[start : end + 1]

def parse_debug_selector(selector: str, n_rows: int) -> list[int]:
    s = (selector or "").strip()
    if not s:
        return list(range(n_rows))

    if s.isdigit():
        k = int(s)
        if k < 1 or k > n_rows:
            raise ValueError(f"DEBUG_SLICE='{s}' out of range 1..{n_rows}")
        return [k - 1]

    if ":" in s:
        parts = s.split(":")
        if len(parts) > 3:
            raise ValueError(f"Invalid slice syntax: '{s}'")

        def to_int(x):
            x = x.strip()
            return None if x == "" else int(x)

        start = to_int(parts[0]) if len(parts) >= 1 else None
        stop  = to_int(parts[1]) if len(parts) >= 2 else None
        step  = to_int(parts[2]) if len(parts) == 3 else None

        sl = slice(start, stop, step)
        return list(range(n_rows))[sl]

    raise ValueError(f"Unrecognized DEBUG_SLICE format: '{selector}'")

def run_y_spec(ws_text: str) -> tuple[dict | None, str, str, int]:
    try:
        proc = subprocess.run(
            ["python", str(Y_SPEC_PY), "--model", MODEL],
            input=ws_text,
            text=True,
            capture_output=True,
            timeout=Y_SPEC_TIMEOUT,
        )
    except subprocess.TimeoutExpired as e:
        stdout = (e.stdout or "").strip()
        stderr = (e.stderr or "TIMEOUT").strip()
        return None, stdout, stderr, 124

    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()

    if proc.returncode != 0:
        return None, stdout, stderr, proc.returncode

    try:
        parsed = json.loads(_extract_json_object(stdout))
        return parsed, stdout, stderr, 0
    except Exception:
        return None, stdout, stderr, 0

def canonicalize_tag(tag: str) -> str:
    """
    Normalize tag strings coming back from the LLM into our canonical vocabulary.
    Returns canonical tag or "" if unknown.
    """
    if not tag:
        return ""

    t = str(tag).strip().lower()

    # Normalize separators / punctuation
    t = t.replace("-", "_").replace(" ", "_")
    t = re.sub(r"[^a-z0-9_]+", "", t)
    t = re.sub(r"_+", "_", t).strip("_")

    # Synonyms / drift mapping (expand as you observe real outputs)
    synonym_map = {
        # none
        "no": "none",
        "null": "none",
        "na": "none",
        "n_a": "none",

        # upheld
        "allow": "upheld",
        "allowed": "upheld",
        "uphold": "upheld",
        "upholding": "upheld",
        "appeal_upheld": "upheld",
        "appeal_allowed": "upheld",

        # verbal warning
        "verbalwarning": "verbal_warning",
        "verbal_warn": "verbal_warning",
        "verbal_warning": "verbal_warning",
        "verbal": "verbal_warning",

        # no contemporaneous evidence
        "no_contemporaneous_record": "no_contemporaneous_evidence",
        "no_contemporaneous_records": "no_contemporaneous_evidence",
        "no_contemporaneous_note": "no_contemporaneous_evidence",
        "no_contemporaneous_notes": "no_contemporaneous_evidence",
        "no_notes": "no_contemporaneous_evidence",
        "no_record": "no_contemporaneous_evidence",
        "no_records": "no_contemporaneous_evidence",
        "no_contemporaneous_evidence": "no_contemporaneous_evidence",

        # predetermination
        "pre_determination": "predetermination",
        "pre_determined": "predetermination",
        "predetermined": "predetermination",
        "predetermination": "predetermination",

        # appeal scope limitation
        "appeal_scope": "appeal_scope_limitation",
        "scope_limitation": "appeal_scope_limitation",
        "appeal_scope_limit": "appeal_scope_limitation",
        "appeal_scope_limitation": "appeal_scope_limitation",
        "outside_scope": "appeal_scope_limitation",
        "out_of_scope": "appeal_scope_limitation",
        "not_in_grounds": "appeal_scope_limitation",
    }

    t = synonym_map.get(t, t)

    return t if t in ALLOWED_TAGS else ""

def make_needle_prompt(text: str) -> str:
    tag_lines = "\n".join([f'- "{t}": {TAG_DEFS[t]}' for t in ALLOWED_TAGS])
    return f"""
TASK:
Given TEXT, select all applicable TAGS from the allowed list.

CRITICAL TAG RULE:
- Each "tag" value MUST be EXACTLY one of the strings in ALLOWED_TAGS (character-for-character).
- Do NOT invent new tag names.
- Do NOT use spaces or hyphens unless they appear in the allowed tag.
- If uncertain, return ONLY ["none"].

ALLOWED_TAGS:
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from TEXT (max 200 chars)
  - negated: true if TEXT explicitly indicates the opposite
- If you cannot quote evidence from TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

TEXT:
<<<
{text.strip()}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()

def run_needle_tagger(text: str) -> tuple[list[dict], str, str, int]:
    try:
        proc = subprocess.run(
            OLLAMA_CMD,
            input=make_needle_prompt(text),
            text=True,
            capture_output=True,
            timeout=NEEDLE_TIMEOUT,
        )
    except subprocess.TimeoutExpired as e:
        stdout = (e.stdout or "").strip()
        stderr = (e.stderr or "TIMEOUT").strip()
        return [], stdout, stderr, 124

    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()

    if proc.returncode != 0:
        return [], stdout, stderr, proc.returncode

    # Default fallback to NONE (keeps pipeline moving, avoids tag drift poisoning)
    def _none():
        return ([{"tag": "none", "confidence": 1.0, "negated": False, "evidence_quote": ""}], stdout, stderr, 0)

    try:
        out = json.loads(_extract_json_object(stdout))
        selected = out.get("selected", [])
        if not isinstance(selected, list):
            return _none()

        cleaned: list[dict] = []
        for item in selected:
            if not isinstance(item, dict):
                continue

            raw_tag = item.get("tag")
            tag = canonicalize_tag(raw_tag)

            # Drop unknown tags entirely (LLM drift)
            if not tag:
                continue

            # Coerce fields safely
            conf = item.get("confidence")
            try:
                conf_f = float(conf) if conf is not None else 0.0
            except Exception:
                conf_f = 0.0
            conf_f = max(0.0, min(1.0, conf_f))

            neg = bool(item.get("negated") is True)
            quote = (item.get("evidence_quote") or "")
            quote = str(quote)[:200]

            cleaned.append(
                {"tag": tag, "confidence": conf_f, "negated": neg, "evidence_quote": quote}
            )

        # If nothing survived, treat as none
        if not cleaned:
            return _none()

        tags = [d["tag"] for d in cleaned if d.get("tag")]

        # Enforce "none" must be alone
        if "none" in tags:
            return _none()

        # De-dup by tag (keep max confidence)
        best = {}
        for d in cleaned:
            t = d["tag"]
            if t == "none":
                continue
            if (t not in best) or (d["confidence"] > best[t]["confidence"]):
                best[t] = d

        final = list(best.values()) if best else [{"tag": "none", "confidence": 1.0, "negated": False, "evidence_quote": ""}]
        return final, stdout, stderr, 0

    except Exception:
        return _none()

def flatten_selected(selected: list[dict]) -> dict:
    rec = {}
    for t in ALLOWED_TAGS:
        if t == "none":
            continue
        rec[f"has__{t}"] = False
        rec[f"conf__{t}"] = 0.0
        rec[f"quote__{t}"] = ""

    for item in selected:
        tag = item.get("tag")
        if not tag or tag == "none":
            continue
        if item.get("negated") is True:
            continue

        rec[f"has__{tag}"] = True
        rec[f"conf__{tag}"] = float(item.get("confidence") or 0.0)
        rec[f"quote__{tag}"] = (item.get("evidence_quote") or "")[:200]

    return rec

# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(CSV_PATH)

if TEXT_COL not in df.columns:
    raise KeyError(f"Column '{TEXT_COL}' not found. Found: {list(df.columns)}")

texts_all = df[TEXT_COL].fillna("").astype(str).tolist()
n_total = len(texts_all)

idxs = parse_debug_selector(DEBUG_SLICE, n_total) if DEBUG else list(range(n_total))

print(f"Total rows in CSV: {n_total}")
print(f"Processing indices (0-based): {idxs[:30]}{' ...' if len(idxs) > 30 else ''}")
print(f"Count: {len(idxs)} | DEBUG={DEBUG} | DEBUG_SLICE='{DEBUG_SLICE}'")

# =========================================================
# RUN PIPELINE
# =========================================================
enrich_by_idx = {}
y_results = {
    "version": "Y_inferred_v2",
    "source": {"csv": str(CSV_PATH), "text_col": TEXT_COL, "model": MODEL},
    "rows": {}
}

for idx in tqdm(idxs, desc="Rows (Needles + y_spec)"):
    ws_text = texts_all[idx].strip()
    X1 = idx + 1

    if not ws_text:
        empty_selected = [{"tag": "none", "confidence": 1.0, "negated": False, "evidence_quote": ""}]
        rec = {
            "ws_len": 0,
            "needle_selected_raw": json.dumps(empty_selected, ensure_ascii=False),
            "needle_rc": 0,
            "y_ok": False,
            "y_rc": 0,
        }
        rec.update(flatten_selected(empty_selected))
        enrich_by_idx[idx] = rec
        continue

    print(f"X1={X1} doc={CSV_PATH.name} | stage=needle")

    t0 = time.time()
    selected, n_stdout, n_stderr, n_rc = run_needle_tagger(ws_text)
    needle_secs = round(time.time() - t0, 2)

    print(f"X1={X1} doc={CSV_PATH.name} | stage=y_spec | needle_secs={needle_secs}")

    t1 = time.time()
    y_json, y_stdout, y_stderr, y_rc = run_y_spec(ws_text)
    y_secs = round(time.time() - t1, 2)

    y_results["rows"][f"X1_{X1:04d}"] = {
        "row_index_1based": X1,
        "doc": CSV_PATH.name,
        "y_ok": bool(y_json),
        "y": y_json if y_json else None,
        "y_returncode": y_rc,
        "y_stderr_head": (y_stderr[:400] if DEBUG and y_stderr else ""),
        "y_stdout_head": (y_stdout[:400] if DEBUG and y_stdout else ""),
        "timing": {"needle_secs": needle_secs, "y_secs": y_secs},
    }

    rec = {
        "ws_len": len(ws_text),
        "needle_selected_raw": json.dumps(selected, ensure_ascii=False),
        "needle_rc": n_rc,
        "y_ok": bool(y_json),
        "y_rc": y_rc,
    }
    rec.update(flatten_selected(selected))
    enrich_by_idx[idx] = rec

# =========================================================
# MERGE + WRITE
# =========================================================
enrich_df = pd.DataFrame.from_dict(enrich_by_idx, orient="index")
enrich_df.index.name = "row_index_0based"

df_out = df.copy().join(enrich_df, how="left")
df_out.insert(0, "X1", df_out.index + 1)
df_out.insert(1, "doc", CSV_PATH.name)

df_out.to_csv(OUT_ENHANCED_CSV, index=False)
OUT_Y_JSON.write_text(json.dumps(y_results, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nWrote outputs:")
print(f" - Enhanced CSV: {OUT_ENHANCED_CSV}")
print(f" - Y_inferred.json: {OUT_Y_JSON}")

display(df_out.head(25))

Total rows in CSV: 12
Processing indices (0-based): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Count: 12 | DEBUG=False | DEBUG_SLICE='3'


Rows (Needles + y_spec):   0%|          | 0/12 [00:00<?, ?it/s]

X1=1 doc=Leonardo_WS.csv | stage=needle
X1=1 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.89
X1=2 doc=Leonardo_WS.csv | stage=needle
X1=2 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.08
X1=3 doc=Leonardo_WS.csv | stage=needle
X1=3 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=1.92
X1=4 doc=Leonardo_WS.csv | stage=needle
X1=4 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=1.82
X1=5 doc=Leonardo_WS.csv | stage=needle
X1=5 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=0.99
X1=6 doc=Leonardo_WS.csv | stage=needle
X1=6 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.12
X1=7 doc=Leonardo_WS.csv | stage=needle
X1=7 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.15
X1=8 doc=Leonardo_WS.csv | stage=needle
X1=8 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=3.07
X1=9 doc=Leonardo_WS.csv | stage=needle
X1=9 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.42
X1=10 doc=Leonardo_WS.csv | stage=needle
X1=10 doc=Leonardo_WS.csv | stage=y_spec | needle_secs=2.43
X1=11 do

,X1,doc,source_row,section_title,section_summary,text_verbatim,claims,evidence_mentioned,evidence_to_request,evidence_to_locate_own,...,quote__verbal_warning,has__no_contemporaneous_evidence,conf__no_contemporaneous_evidence,quote__no_contemporaneous_evidence,has__predetermination,conf__predetermination,quote__predetermination,has__appeal_scope_limitation,conf__appeal_scope_limitation,quote__appeal_scope_limitation
0,1,Leonardo_WS.csv,1,"Performance Assessment, Role Evolution, and Ma...","Between 2019 and 2020, the Claimant’s role was...","Background, Role Evolution, Performance Contex...","[\n {\n ""text"": ""Throughout the Claimant’s...","[\n ""foundational training"",\n ""queue re...",[],[],...,,True,0.8,"From 2022 onwards, project work was a required...",False,0.0,,True,0.7,This change was explicit in the structure and ...
1,2,Leonardo_WS.csv,2,"Definition of Projects, Management Instruction...",Management expressly defined and endorsed the ...,"Definition of “Projects”, Management Instructi...",['Management raised system behaviour in July 2...,• DSAR internal management communications (Jan...,• Any internal policy or guidance warning agai...,• Claimant’s calendar or work logs evidencing ...,...,,False,0.0,,False,0.0,,False,0.0,
2,3,Leonardo_WS.csv,3,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,Escalation Only After Role Change (February 20...,[],[],[],[],...,,True,0.9,the absence of any contemporaneous instruction...,True,0.8,the disciplinary process relied on system data...,False,0.0,
3,4,Leonardo_WS.csv,4,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,19–20 March 2024: First Presentation of Allega...,[{'text': 'The meeting on 19 March 2024 was no...,"[""calendar invitation titled 'meeting'"", 'meet...",[],[],...,,False,0.0,,False,0.0,,False,0.0,
4,5,Leonardo_WS.csv,5,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,"Reassurance, Reliance, and Subsequent Conduct\...",['Ms Oteri provided reassurance on 20 March 20...,"['Statement by Ms Oteri on 20 March 2024', 'In...",[],[],...,,False,0.0,,False,0.0,,False,0.0,
5,6,Leonardo_WS.csv,6,Analysis Omissions,Respondent’s analysis fails to distinguish bet...,"Failure to Distinguish Queue Time, Absence of ...",[{'text': 'A critical omission in the Responde...,"['crash figures', 'disciplinary hearing', 'app...","['underlying system logs', 'detailed logs show...",[],...,,False,0.0,,False,0.0,,False,0.0,
6,7,Leonardo_WS.csv,7,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,Notice of Disciplinary Hearing and Procedural ...,[{'text': 'The author received very short noti...,['chat message from Ms Parisi on 2 April 2024'...,"['ET3 document', 'crash data', 'contemporaneou...",['chat message from Ms Parisi on 2 April 2024'...,...,,False,0.0,,True,0.7,"Despite this, my defence was not properly enga...",True,0.8,"Despite this, my defence was not properly enga..."
7,8,Leonardo_WS.csv,8,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,"The Disciplinary Hearing: Hostile Conduct, Lin...",[{'text': 'Mr O’Hagan interrupted and prevente...,"['Written defence submitted the previous day',...","[""Documentary evidence or analysis supporting ...",['Any records or notes from the speaker regard...,...,,True,0.8,"No such verification was produced. Instead, it...",True,0.9,It later became apparent that the hearing foll...,False,0.0,
8,9,Leonardo_WS.csv,9,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,Failure to Comply with the Respondent’s Own Di...,['The Respondent’s disciplinary policy was not...,"['Respondent’s disciplinary policy', 'Chat mes...",['Details of the Respondent’s disciplinary pol...,['Any personal records or notes related to the...,...,,True,0.9,I also strongly contested the Respondent’s cha...,False,0.0,,True,0.7,Despite the seriousness of the allegations lat...
9,10,Leonardo_WS.csv,10,Dismissal Details,Respondent d